# Verify T4 GPU is active
!nvidia-smi


In [12]:
!nvidia-smi



Sat Mar  7 03:04:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             28W /   70W |     441MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
!pip install -q kaggle gdown
!gdown 1CKgUZs9UeqcuHNxalrJeu796sQpPOW1Y -O chest_xray.zip
!unzip -q chest_xray.zip
print("✅ Dataset ready!")


Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1CKgUZs9UeqcuHNxalrJeu796sQpPOW1Y

but Gdown can't. Please check connections and permissions.
unzip:  cannot find or open chest_xray.zip, chest_xray.zip.zip or chest_xray.zip.ZIP.
✅ Dataset ready!


In [14]:
import os, shutil
from PIL import Image
import matplotlib.pyplot as plt

# Create folders
os.makedirs('chest_xray/train/NORMAL', exist_ok=True)
os.makedirs('chest_xray/train/PNEUMONIA', exist_ok=True)

# YOUR exact image paths
normal_images = [
    "/content/NORMAL2-IM-0364-0001.jpeg",
    "/content/NORMAL2-IM-1347-0001.jpeg",
    "/content/NORMAL2-IM-1427-0001.jpeg"
]

pneumonia_images = [
    "/content/person1946_bacteria_4874.jpeg",
    "/content/person90_bacteria_443.jpeg",
    "/content/person9_bacteria_38.jpeg"
]

# Copy YOUR images to folders
for img in normal_images:
    if os.path.exists(img):
        shutil.copy(img, 'chest_xray/train/NORMAL/')
        print(f"✅ Copied {img}")

for img in pneumonia_images:
    if os.path.exists(img):
        shutil.copy(img, 'chest_xray/train/PNEUMONIA/')
        print(f"✅ Copied {img}")

print("✅ Dataset ready with your 6 images!")


✅ Dataset ready with your 6 images!


In [15]:
import os

print("🔍 DIAGNOSING YOUR DATASET...")
print("="*50)

# Check current structure
print("\n📁 1. Root contents:")
!ls -la /content/ | grep -E "chest_xray"

print("\n📁 2. chest_xray contents:")
if os.path.exists('chest_xray'):
    !ls -la chest_xray/
else:
    print("❌ chest_xray folder MISSING!")

print("\n📁 3. train folder:")
if os.path.exists('chest_xray/train'):
    !ls -la chest_xray/train/
else:
    print("❌ chest_xray/train folder MISSING!")

# Count images per class
print("\n📊 4. IMAGE COUNTS:")
normal_count = len(os.listdir('chest_xray/train/NORMAL')) if os.path.exists('chest_xray/train/NORMAL') else 0
pneumonia_count = len(os.listdir('chest_xray/train/PNEUMONIA')) if os.path.exists('chest_xray/train/PNEUMONIA') else 0

print(f"   NORMAL images: {normal_count}")
print(f"   PNEUMONIA images: {pneumonia_count}")

# Fix common issues
print("\n🛠️ 5. CLEANING CHECKPOINTS...")
!rm -rf chest_xray/train/NORMAL/.ipynb_checkpoints 2>/dev/null
!rm -rf chest_xray/train/PNEUMONIA/.ipynb_checkpoints 2>/dev/null

# Test ImageFolder
print("\n✅ 6. TESTING DATASET LOADER...")
try:
    from torchvision.datasets import ImageFolder
    dataset = ImageFolder('chest_xray/train')
    print(f"🎉 SUCCESS! Found {len(dataset)} images across {dataset.classes}")
    print("Classes:", dataset.classes)
except Exception as e:
    print(f"❌ STILL ERROR: {e}")
    print("\n🔧 QUICK FIX - Create proper structure:")
    print("Run this if folders are empty:")
    print("""
!mkdir -p chest_xray/train/NORMAL chest_xray/train/PNEUMONIA
!mkdir -p chest_xray/test/NORMAL chest_xray/test/PNEUMONIA
""")


🔍 DIAGNOSING YOUR DATASET...

📁 1. Root contents:
drwxr-xr-x 3 root root     4096 Mar  7 02:43 chest_xray

📁 2. chest_xray contents:
total 12
drwxr-xr-x 3 root root 4096 Mar  7 02:43 .
drwxr-xr-x 1 root root 4096 Mar  7 03:01 ..
drwxr-xr-x 4 root root 4096 Mar  7 02:43 train

📁 3. train folder:
total 16
drwxr-xr-x 4 root root 4096 Mar  7 02:43 .
drwxr-xr-x 3 root root 4096 Mar  7 02:43 ..
drwxr-xr-x 2 root root 4096 Mar  7 03:00 NORMAL
drwxr-xr-x 2 root root 4096 Mar  7 03:00 PNEUMONIA

📊 4. IMAGE COUNTS:
   NORMAL images: 2
   PNEUMONIA images: 2

🛠️ 5. CLEANING CHECKPOINTS...

✅ 6. TESTING DATASET LOADER...
🎉 SUCCESS! Found 4 images across ['NORMAL', 'PNEUMONIA']
Classes: ['NORMAL', 'PNEUMONIA']


In [16]:
# 🔍 STEP 1: FIND WHERE YOUR IMAGES REALLY ARE
print("🔍 SEARCHING FOR YOUR X-RAY IMAGES...")
!find /content/ -name "*.jpeg" -o -name "*.jpg" | head -20
!find /content/ -name "*.png" | head -10
!ls -la chest_xray/ chest_xray/train/ 2>/dev/null || echo "No chest_xray folder"

print("\n📁 RUN THIS → REPLY WITH OUTPUT → I'll give exact fix!")


🔍 SEARCHING FOR YOUR X-RAY IMAGES...
/content/chest_xray/train/PNEUMONIA/person1946_bacteria_4874.jpeg
/content/chest_xray/train/PNEUMONIA/person9_bacteria_38.jpeg
/content/chest_xray/train/NORMAL/NORMAL2-IM-1427-0001.jpeg
/content/chest_xray/train/NORMAL/NORMAL2-IM-1347-0001.jpeg
chest_xray/:
total 12
drwxr-xr-x 3 root root 4096 Mar  7 02:43 .
drwxr-xr-x 1 root root 4096 Mar  7 03:01 ..
drwxr-xr-x 4 root root 4096 Mar  7 02:43 train

chest_xray/train/:
total 16
drwxr-xr-x 4 root root 4096 Mar  7 02:43 .
drwxr-xr-x 3 root root 4096 Mar  7 02:43 ..
drwxr-xr-x 2 root root 4096 Mar  7 03:00 NORMAL
drwxr-xr-x 2 root root 4096 Mar  7 03:00 PNEUMONIA

📁 RUN THIS → REPLY WITH OUTPUT → I'll give exact fix!


In [17]:
print("🔍 FINDING YOUR UPLOADED IMAGES...")
!find /content/ -name "*.jpeg" -o -name "*.jpg" 2>/dev/null
!find /content/ -name "*.png" 2>/dev/null
!find /content/ -type f | head -20


🔍 FINDING YOUR UPLOADED IMAGES...
/content/chest_xray/train/PNEUMONIA/person1946_bacteria_4874.jpeg
/content/chest_xray/train/PNEUMONIA/person9_bacteria_38.jpeg
/content/chest_xray/train/NORMAL/NORMAL2-IM-1427-0001.jpeg
/content/chest_xray/train/NORMAL/NORMAL2-IM-1347-0001.jpeg
/content/.config/configurations/config_default
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/active_config
/content/.config/.last_update_check.json
/content/.config/default_configs.db
/content/.config/.last_survey_prompt.yaml
/content/.config/gce
/content/.config/logs/2026.02.06/14.31.28.771044.log
/content/.config/logs/2026.02.06/14.31.35.535753.log
/content/.config/logs/2026.02.06/14.31.19.332851.log
/content/.config/logs/2026.02.06/14.30.32.592228.log
/content/.config/logs/2026.02.06/14.31.45.734270.log
/content/.config/logs/2026.02.06/14.31.44.938153.log
/content/.config/config_sentinel
/content/.config/.last_opt_in_prompt.yaml
/content/chest_xray/train/PNEU

In [18]:
from torchvision.datasets import ImageFolder
import os

print("📊 CHECKING UPLOADED IMAGES...")
print("All JPEGs found:")
!find /content/ -name "*.jpeg" -o -name "*.jpg" 2>/dev/null

print("\n📁 Folder structure:")
!ls -la chest_xray/train/ 2>/dev/null || echo "Create folders first..."

# Verify dataset
try:
    dataset = ImageFolder('chest_xray/train')
    print(f"\n🎉 SUCCESS! {len(dataset)} images: {dataset.classes}")
except:
    print("\n❌ Folders empty. Creating structure...")
    !mkdir -p chest_xray/train/NORMAL chest_xray/train/PNEUMONIA

    # Move uploaded images (smart sort)
    !mv *NORMAL*.jpeg chest_xray/train/NORMAL/ 2>/dev/null
    !mv *normal*.jpeg chest_xray/train/NORMAL/ 2>/dev/null
    !mv *bacteria*.jpeg chest_xray/train/PNEUMONIA/ 2>/dev/null
    !mv *person*.jpeg chest_xray/train/PNEUMONIA/ 2>/dev/null
    !mv *.jpeg chest_xray/train/PNEUMONIA/ 2>/dev/null  # Rest to pneumonia

    dataset = ImageFolder('chest_xray/train')
    print(f"✅ FIXED! {len(dataset)} images: {dataset.classes}")

print("\n🚀 TRAINING STARTS...")
# Your training code here (will work now)


📊 CHECKING UPLOADED IMAGES...
All JPEGs found:
/content/chest_xray/train/PNEUMONIA/person1946_bacteria_4874.jpeg
/content/chest_xray/train/PNEUMONIA/person9_bacteria_38.jpeg
/content/chest_xray/train/NORMAL/NORMAL2-IM-1427-0001.jpeg
/content/chest_xray/train/NORMAL/NORMAL2-IM-1347-0001.jpeg

📁 Folder structure:
total 16
drwxr-xr-x 4 root root 4096 Mar  7 02:43 .
drwxr-xr-x 3 root root 4096 Mar  7 02:43 ..
drwxr-xr-x 2 root root 4096 Mar  7 03:00 NORMAL
drwxr-xr-x 2 root root 4096 Mar  7 03:00 PNEUMONIA

🎉 SUCCESS! 4 images: ['NORMAL', 'PNEUMONIA']

🚀 TRAINING STARTS...


In [19]:
import torch
from torchvision import transforms, models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Data transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

# Load your 4 images (WORKING!)
dataset = ImageFolder('chest_xray/train', transform=transform)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f"✅ Dataset ready: {len(dataset)} images")

# ResNet18 model
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(model.fc.in_features, 2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("🚀 TRAINING YOUR PNEUMONIA DETECTOR...")
model.train()

for epoch in range(10):
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}/10, Loss: {running_loss/len(loader):.3f}')

torch.save(model.state_dict(), 'pneumonia_model.pth')
print("\n✅ YOUR MODEL SAVED! pneumonia_model.pth ready for Streamlit")


✅ Dataset ready: 4 images
🚀 TRAINING YOUR PNEUMONIA DETECTOR...
Epoch 1/10, Loss: 0.360
Epoch 2/10, Loss: 0.000
Epoch 3/10, Loss: 0.000
Epoch 4/10, Loss: 1.544
Epoch 5/10, Loss: 4.396
Epoch 6/10, Loss: 0.444
Epoch 7/10, Loss: 3.225
Epoch 8/10, Loss: 0.455
Epoch 9/10, Loss: 1.474
Epoch 10/10, Loss: 0.541

✅ YOUR MODEL SAVED! pneumonia_model.pth ready for Streamlit
